In [1]:
#Importing Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
!pip install mlflow

In [3]:
import mlflow

In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

In [5]:
mlflow.set_experiment("HousePricePrediction")

2026/05/26 07:11:39 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/26 07:11:39 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
2026/05/26 07:11:39 INFO mlflow.tracking.fluent: Experiment with name 'HousePricePrediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='/workspaces/mlops_sample/mlruns/1', creation_time=1779779499289, experiment_id='1', last_update_time=1779779499289, lifecycle_stage='active', name='HousePricePrediction', tags={}>

In [6]:
data = pd.read_csv('House Price Prediction Dataset.csv')

In [7]:
data.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         2000 non-null   int64 
 1   Area       2000 non-null   int64 
 2   Bedrooms   2000 non-null   int64 
 3   Bathrooms  2000 non-null   int64 
 4   Floors     2000 non-null   int64 
 5   YearBuilt  2000 non-null   int64 
 6   Location   2000 non-null   object
 7   Condition  2000 non-null   object
 8   Garage     2000 non-null   object
 9   Price      2000 non-null   int64 
dtypes: int64(7), object(3)
memory usage: 156.4+ KB


In [24]:
data.describe()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,1000.500000,2786.209500,3.003500,2.55250,1.993500,1961.446000,537676.855000
std,577.494589,1295.146799,1.424606,1.10899,0.809188,35.926695,276428.845719
min,1.000000,501.000000,1.000000,1.00000,1.000000,1900.000000,50005.000000
25%,500.750000,1653.000000,2.000000,2.00000,1.000000,1930.000000,300098.000000
50%,1000.500000,2833.000000,3.000000,3.00000,2.000000,1961.000000,539254.000000
75%,1500.250000,3887.500000,4.000000,4.00000,3.000000,1993.000000,780086.000000
max,2000.000000,4999.000000,5.000000,4.00000,3.000000,2023.000000,999656.000000


In [9]:
data['Location'].unique()

array(['Downtown', 'Suburban', 'Urban', 'Rural'], dtype=object)

In [10]:
data['Condition'].unique()

array(['Excellent', 'Good', 'Fair', 'Poor'], dtype=object)

In [11]:
data['Garage'].unique()

array(['No', 'Yes'], dtype=object)

In [12]:
final_df  = pd.get_dummies(data = data,drop_first=True, columns=['Location', 'Condition', 'Garage'], dtype='int')

In [13]:
final_df.head()

,Id,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price,Location_Rural,Location_Suburban,Location_Urban,Condition_Fair,Condition_Good,Condition_Poor,Garage_Yes
0,1,1360,5,4,3,1970,149919,0,0,0,0,0,0,0
1,2,4272,5,4,3,1958,424998,0,0,0,0,0,0,0
2,3,3592,2,2,3,1938,266746,0,0,0,0,1,0,0
3,4,966,4,2,2,1902,244020,0,1,0,1,0,0,1
4,5,4926,1,4,2,1975,636056,0,0,0,1,0,0,1


In [19]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

In [14]:
features = final_df[final_df.columns.drop(['Id','Price', 'YearBuilt'])].values

In [15]:
target = final_df['Price'].values

In [16]:
features

array([[1360,    5,    4, ...,    0,    0,    0],
       [4272,    5,    4, ...,    0,    0,    0],
       [3592,    2,    2, ...,    1,    0,    0],
       ...,
       [1062,    5,    1, ...,    0,    1,    0],
       [4062,    3,    1, ...,    0,    0,    1],
       [2989,    5,    1, ...,    0,    0,    0]])

In [17]:
target

array([149919, 424998, 266746, ..., 476925, 161119, 482525])

In [20]:
X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.20, random_state=42)

In [21]:
training_data_path = final_df.to_csv('cleaned_data.csv')

# Running Different Models

In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

In [25]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [23]:
##  Multiple Linear Regression

In [26]:
with mlflow.start_run():
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    linear_reg = LinearRegression()
    linear_reg.fit(X_train, y_train)
    y_prediction = linear_reg.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    r2_score = r2_score(y_true=y_test, y_pred=y_prediction)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2_score", r2_score)

In [30]:
with mlflow.start_run(run_name="Polynomial Regression"):
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    mlflow.log_param("algorithm", "Polynomial Function")
    mlflow.log_param("polynomial_degree", "4")
    poly = PolynomialFeatures(degree=4, include_bias=False)
    X_poly = poly.fit_transform(X_train)
    model = LinearRegression()
    model.fit(X_poly, y_train)
    y_prediction = model.predict(poly.fit_transform(X_test))
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    mlflow.log_metric("rmse", rmse)

In [31]:
!pip install xgboost

In [32]:
from xgboost import XGBRegressor

In [33]:
xgb = XGBRegressor()

In [36]:
with mlflow.start_run(run_name="XGB Regressor default"):
    mlflow.set_tag("developer", "Praveen")
    mlflow.log_param("algorithm", "XGB Regressor with default Params")
    mlflow.log_param("data", "/workspaces/mlops_sample/cleaned_data.csv")
    xgb.fit(X_train, y_train)
    y_prediction = xgb.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_true=y_test, y_pred=y_prediction))
    mlflow.log_metric("rmse", rmse)